# Découverte Sentinel-2 autour de Libreville

**Gabon GeoAI Lab - Observer · Comprendre · Prédire · Simuler**

Objectif : rechercher des produits et comparer leurs métadonnées. Aucun téléchargement d’image, aucune authentification, aucune clé ni écriture de données.

Dans VS Code, installer les extensions Python et Jupyter, puis choisir le noyau Python de `.venv` (3.10.11 ou 3.12). Depuis la racine du dépôt :

```bash
python -m pip install -r requirements-dev.txt
```

Exécuter les cellules dans l’ordre. Une connexion Internet est nécessaire.


## 1. Comprendre le catalogue

STAC (SpatioTemporal Asset Catalog) décrit des données associées à un lieu et à une date. Un catalogue contient des **collections** ; chaque produit est un **Item**. Les **assets** désignent les fichiers associés, que nous ne consultons ni ne téléchargeons ici.

Sentinel-2 **Level-2A (L2A)** fournit notamment une réflectance de surface après correction atmosphérique. Un produit contient plusieurs bandes et résolutions ; ce notebook n’effectue aucun traitement raster.

Sources officielles : [documentation STAC CDSE](https://documentation.dataspace.copernicus.eu/APIs/STAC.html), [catalogue public](https://stac.dataspace.copernicus.eu/v1/).


In [72]:
from datetime import date
from math import isfinite
from urllib.parse import urljoin, urlparse

import requests

STAC_URL = "https://stac.dataspace.copernicus.eu/v1/"
COLLECTION_ID = "sentinel-2-l2a"
TIMEOUT_SECONDS = 40


## 2. Définir la zone pilote et la période

Une **bounding box** est un rectangle : `[longitude minimale, latitude minimale, longitude maximale, latitude maximale]`, soit ouest, sud, est, nord. Les coordonnées sont en degrés dans le référentiel géographique **WGS84**, avec la longitude en premier.

Notre rectangle va de 9,25° à 9,65° Est et de 0,20° à 0,65° Nord. Il couvre Libreville et ses environs, avec une partie côtière et maritime. C’est une zone expérimentale, pas une limite administrative.

Les produits recherchés **intersectent** ce rectangle ; cela ne garantit pas qu’un seul produit le couvre entièrement. Modifier les paramètres ci-dessous, puis réexécuter les cellules suivantes.

Le seuil nuageux sera appliqué localement aux métadonnées. La période est exprimée en UTC ; la date de fin comprend toute la journée.


In [73]:
LIBREVILLE_BBOX = [9.25, 0.20, 9.65, 0.65]
START_DATE = "2025-01-01"
END_DATE = "2025-01-31"
MAX_CLOUD_COVER = 40.0  # Pourcentage entre 0 et 100.
TOP_N = 10
PAGE_SIZE = 100
MAX_PAGES = 10  # Limite pédagogique pour éviter une recherche illimitée.


In [74]:
west, south, east, north = LIBREVILLE_BBOX
if not (-180 <= west < east <= 180 and -90 <= south < north <= 90):
    raise ValueError("Bounding box invalide : vérifier longitude et latitude.")
if date.fromisoformat(START_DATE) > date.fromisoformat(END_DATE):
    raise ValueError("La date de début doit précéder la date de fin.")
if not 0 <= MAX_CLOUD_COVER <= 100:
    raise ValueError("Le seuil nuageux doit être compris entre 0 et 100.")
if TOP_N < 1 or PAGE_SIZE < 1 or MAX_PAGES < 1:
    raise ValueError("Les limites doivent être positives.")

search_period = f"{START_DATE}T00:00:00Z/{END_DATE}T23:59:59.999Z"
print("Zone WGS84 :", LIBREVILLE_BBOX)
print("Période UTC :", search_period)


Zone WGS84 : [9.25, 0.2, 9.65, 0.65]
Période UTC : 2025-01-01T00:00:00Z/2025-01-31T23:59:59.999Z


## 3. Vérifier la collection publique

Une requête HTTP GET lit un document JSON. `raise_for_status()` rend explicites les erreurs du serveur ; le délai limite l’attente. Aucune information d’identification n’est fournie. En cas d’échec réseau, corriger la connexion puis relancer la cellule.


In [75]:
# Utilise le magasin de certificats du système d'exploitation.
# Utile notamment sur les postes d'entreprise utilisant leurs propres
# autorités de certification pour les connexions HTTPS.
import truststore

truststore.inject_into_ssl()

In [76]:
response = requests.get(
    f"{STAC_URL}collections/{COLLECTION_ID}", timeout=TIMEOUT_SECONDS
)
response.raise_for_status()
collection = response.json()
if collection.get("id") != COLLECTION_ID:
    raise ValueError("La collection attendue n’a pas été retournée.")
print("Collection :", collection["id"])
print("Titre :", collection.get("title", "Non renseigné"))


Collection : sentinel-2-l2a
Titre : Sentinel-2 Level-2A


## 4. Rechercher les métadonnées

`collections`, `bbox` et `datetime` limitent la recherche. `limit` fixe la taille demandée d’une page. Lorsqu’un lien STAC `next` est présent, nous suivons les pages suivantes jusqu’à la limite choisie.

Seules les métadonnées JSON du catalogue sont lues. Aucun lien d’asset n’est ouvert. Le tri final porte sur les résultats récupérés ; une recherche interrompue ou un catalogue partiel ne donne pas un classement exhaustif.


In [77]:
search_params = {
    "collections": COLLECTION_ID,
    "bbox": ",".join(str(value) for value in LIBREVILLE_BBOX),
    "datetime": search_period,
    "limit": PAGE_SIZE,
}
items_by_id = {}
next_url = f"{STAC_URL}search"
params = search_params
search_truncated = False

for page_number in range(1, MAX_PAGES + 1):
    response = requests.get(next_url, params=params, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    page = response.json()
    if page.get("type") != "FeatureCollection":
        raise ValueError("La réponse n’est pas une collection de résultats STAC.")
    features = page.get("features", [])
    for item in features:
        items_by_id[item["id"]] = item
    print(f"Page {page_number} : {len(features)} produit(s)")

    next_link = next(
        (link for link in page.get("links", []) if link.get("rel") == "next"), None
    )
    if next_link is None:
        if len(features) >= PAGE_SIZE:
            print("Attention : page pleine sans lien suivant ; exhaustivité à vérifier.")
        break
    if page_number == MAX_PAGES:
        search_truncated = True
        break
    next_url = urljoin(response.url, next_link["href"])
    # Rester sur le catalogue public et sur une pagination GET.
    if (urlparse(next_url).scheme != "https"
            or urlparse(next_url).netloc != urlparse(STAC_URL).netloc
            or next_link.get("method", "GET").upper() != "GET"):
        raise ValueError("Lien de pagination non pris en charge : examiner les métadonnées.")
    params = None  # Le lien suivant porte ses propres paramètres.

items = list(items_by_id.values())
print(f"Total : {len(items)} produit(s) distinct(s)")
if search_truncated:
    print("Recherche limitée : réduire la période ou augmenter MAX_PAGES avant comparaison.")


Page 1 : 5 produit(s)
Total : 5 produit(s) distinct(s)


## 5. Lire et afficher les résultats

`datetime` indique la date d’acquisition et `eo:cloud_cover` la couverture nuageuse en pourcentage : une valeur faible est généralement préférable pour observer la surface.

Cette valeur concerne **l’ensemble du produit**, pas uniquement notre rectangle. Elle ne suffit pas à déterminer la qualité locale : ombres, brume, couverture spatiale et masques de qualité devront être examinés dans une phase ultérieure.

Les valeurs absentes ou invalides sont affichées comme « Non renseignée » et exclues des candidates. Aucune valeur manquante n’est assimilée à 0 %.


In [78]:
def read_cloud_cover(value):
    """Retourne un pourcentage valide ou None si la métadonnée manque."""
    if value is None or isinstance(value, bool):
        return None
    try:
        cloud_cover = float(value)
    except (TypeError, ValueError):
        return None
    return cloud_cover if isfinite(cloud_cover) and 0 <= cloud_cover <= 100 else None


records = []
for item in items:
    properties = item.get("properties", {})
    records.append({
        "id": item["id"],
        "acquired_at": properties.get("datetime"),
        "cloud_cover": read_cloud_cover(properties.get("eo:cloud_cover")),
    })


In [79]:
def display_records(rows):
    """Affiche un tableau texte sans dépendance supplémentaire."""
    if not rows:
        print("Aucun résultat pour ces paramètres.")
        return
    print(f"{'ID':<75} | {'Acquisition UTC':<27} | Nuages (%)")
    print("-" * 120)
    for row in rows:
        cloud = row["cloud_cover"]
        cloud_label = f"{cloud:.2f}" if cloud is not None else "Non renseignée"
        acquired_at = row["acquired_at"] or "Non renseignée"
        print(f"{row['id']:<75} | {acquired_at:<27} | {cloud_label}")


display_records(records)


ID                                                                          | Acquisition UTC             | Nuages (%)
------------------------------------------------------------------------------------------------------------------------
S2B_MSIL2A_20250125T093159_N0511_R136_T32NNF_20250125T115658                | 2025-01-25T09:31:59.024Z    | 36.98
S2A_MSIL2A_20250120T093321_N0511_R136_T32NNF_20250120T125148                | 2025-01-20T09:33:21.024Z    | 82.64
S2B_MSIL2A_20250115T093239_N0511_R136_T32NNF_20250115T120528                | 2025-01-15T09:32:39.024Z    | 36.14
S2A_MSIL2A_20250110T093351_N0511_R136_T32NNF_20250110T135958                | 2025-01-10T09:33:51.024Z    | 99.41
S2B_MSIL2A_20250105T093309_N0511_R136_T32NNF_20250105T120229                | 2025-01-05T09:33:09.024Z    | 99.98


## 6. Classer les meilleures candidates

Nous conservons les produits dont la couverture nuageuse est renseignée et inférieure ou égale au seuil. Le classement privilégie la couverture nuageuse croissante, puis la date croissante et l’ID pour départager les égalités.

Il s’agit d’un classement des **produits** : plusieurs produits peuvent appartenir à une même acquisition. Il ne sélectionne pas encore une mosaïque couvrant toute la zone.


In [80]:
candidates = [
    row for row in records
    if row["cloud_cover"] is not None and row["cloud_cover"] <= MAX_CLOUD_COVER
]
ranked_candidates = sorted(
    candidates,
    key=lambda row: (row["cloud_cover"], row["acquired_at"] or "", row["id"]),
)
missing_cloud_count = sum(row["cloud_cover"] is None for row in records)
print(f"Seuil : {MAX_CLOUD_COVER:.1f} % ; candidates : {len(ranked_candidates)}")
print(f"Couverture nuageuse absente ou invalide : {missing_cloud_count}")
display_records(ranked_candidates[:TOP_N])


Seuil : 40.0 % ; candidates : 2
Couverture nuageuse absente ou invalide : 0
ID                                                                          | Acquisition UTC             | Nuages (%)
------------------------------------------------------------------------------------------------------------------------
S2B_MSIL2A_20250115T093239_N0511_R136_T32NNF_20250115T120528                | 2025-01-15T09:32:39.024Z    | 36.14
S2B_MSIL2A_20250125T093159_N0511_R136_T32NNF_20250125T115658                | 2025-01-25T09:31:59.024Z    | 36.98


## Pour apprendre et vérifier

1. Réexécuter toutes les cellules avec les paramètres par défaut et lire les trois colonnes.
2. Réduire le seuil nuageux à 20 %, puis relancer le classement : le nombre de candidates ne doit pas augmenter.
3. Changer le mois et relancer toutes les cellules : comparer la disponibilité et les nuages.
4. Vérifier que les candidates respectent le seuil et que la couverture nuageuse est croissante.
5. Si aucun produit ne passe le seuil, élargir la période ou relever le seuil explicitement.

Avant de partager le notebook, effacer ses sorties. Aucun fichier d’image, donnée raster ou secret ne doit être ajouté au dépôt.
